# Phase 8 — Does It Survive Real Trading Costs?

**The idea in one line.** Every number so far ignores the tolls of actually trading: broker
commission and the fact that you buy at the ask and sell at the bid. Here we charge a realistic toll
on every pound that changes hands in the Phase 6 walk-forward and ask the only question that matters
to a practitioner: **is there anything left?**

**Why pairs trading is especially exposed to costs.** The strategy trades often (every entry and exit
is *two* stock trades — one long, one short) and its edge per trade is modest. Plenty of published
pairs-trading results die the moment costs are applied — Do & Faff (2012) showed exactly this for the
classical distance strategy. Charging costs honestly is therefore not a formality; it is where the
practical-viability claim is won or lost.

## The cost model (two components, per pound traded)

- **Commission — 2 basis points** (0.02%). Institutional brokerage on US large-caps is typically
  1–2 bp of the amount traded; we take the top of that range.
- **Slippage — 5 basis points** (0.05%). You cross half the bid–ask spread each time you trade, plus
  a little market impact. Quoted spreads on S&P 500 names average a few basis points; 5 bp is a
  deliberately unflattering estimate for the modest sizes this strategy implies.

**Total: 7 bp per side.** Phase 6 already recorded exactly how much capital changed hands each day
(`wf_turnover.csv` — entries, exits, and the forced quarter-end closes all included), so applying the
model is one line: each day's cost = 7 bp × that day's turnover. Because we doubt any single number,
we also show half (3.5 bp — cheap electronic execution) and double (14 bp — a punitive stress case).

*Technical note (safe to skip):* costs are charged on turnover measured as a fraction of portfolio
capital, the same units as the daily returns, so subtraction is exact — no share counts needed.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from walkforward import perf

wf_ret = pd.read_csv('data/model_input/wf_returns.csv', index_col=0, parse_dates=True)['ret']
wf_tvr = pd.read_csv('data/model_input/wf_turnover.csv', index_col=0, parse_dates=True)['turnover']

COMM_BPS, SLIP_BPS = 2.0, 5.0
COST_BPS = COMM_BPS + SLIP_BPS

net_ret = wf_ret - COST_BPS / 1e4 * wf_tvr
m_gross, eq_gross = perf(wf_ret)
m_net, eq_net = perf(net_ret)

total_costs = (COST_BPS / 1e4 * wf_tvr).sum()
print(f'Total capital turned over: {wf_tvr.sum():.1f}x  |  total costs paid: {total_costs:.2%} of capital')
print()
side = pd.concat([pd.Series(m_gross, name='gross'), pd.Series(m_net, name='net')], axis=1)
side.loc['cost_drag_per_year'] = [0, (m_gross['ann_return'] - m_net['ann_return'])]
print(side.round(4).to_string())

## The picture, and the stress test

**Left — gross vs net.** The same walk-forward pot with and without the toll. The vertical gap
between the lines is the cumulative cost of trading; what matters is whether the net line still ends
above £1 and still trends upward.

**Right — the stress test.** The net reward-for-risk (Sharpe) at half, headline, and double costs.
A strategy that only works at the cheapest assumption is fragile; one that stays positive even at
14 bp has a real margin of safety.

In [ ]:
sens = {}
for bps in [COST_BPS / 2, COST_BPS, COST_BPS * 2]:
    m_s, _ = perf(wf_ret - bps / 1e4 * wf_tvr)
    sens[f'{bps:.1f} bp'] = m_s
sens = pd.DataFrame(sens).T

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={'width_ratios': [2, 1]})
ax[0].plot(eq_gross.index, eq_gross.values, color='navy', label='gross')
ax[0].plot(eq_net.index, eq_net.values, color='darkorange', label=f'net ({COST_BPS:.0f} bp/side)')
ax[0].axhline(1, color='grey', ls='--', lw=0.7); ax[0].legend()
ax[0].set_title('Walk-forward equity: gross vs net of costs')
ax[1].bar(sens.index, sens.sharpe, color=['seagreen', 'steelblue', 'indianred'])
ax[1].axhline(0, color='k', lw=0.6)
ax[1].set_title('Net Sharpe at each cost level')
plt.tight_layout(); plt.show()

print(sens.round(4).to_string())

## The verdict, in plain words

The cell below prints the sentence the dissertation needs — whether the strategy remains profitable
after realistic friction, and how much of the gross return the tolls consumed. Report it as printed,
favourable or not: a strategy that dies at 7 bp is a legitimate (and publishable) finding about the
limits of the approach, exactly as Do & Faff's was.

In [ ]:
print(f"Gross cumulative return : {m_gross['cum_return']:+.1%}")
print(f"Net cumulative return   : {m_net['cum_return']:+.1%}   (Sharpe {m_net['sharpe']:.2f})")
print(f"Costs consumed          : {total_costs:.2%} of capital")
if m_gross['cum_return'] > 0:
    print(f"Share of gross profit lost to costs: {1 - m_net['cum_return'] / m_gross['cum_return']:.0%}")
verdict = 'SURVIVES' if m_net['cum_return'] > 0 else 'DOES NOT SURVIVE'
print(f"At {COST_BPS:.0f} bp per side the walk-forward strategy {verdict} realistic transaction costs.")

## Save the hand-off files

- **wf_net_returns.csv** — the daily net-of-cost walk-forward returns (Phase 9 benchmarks all
  variants net-of-costs; Phase 10 draws the final gross-vs-net figure).
- **cost_summary.csv** — the cost assumptions and the gross/net scorecards in one table. Phase 9
  reads the headline `cost_bps` from here so every pipeline variant is charged the *same* toll —
  one source of truth.

In [ ]:
net_ret.rename('ret').to_csv('data/model_input/wf_net_returns.csv')
summary = pd.Series({'commission_bps': COMM_BPS, 'slippage_bps': SLIP_BPS, 'cost_bps': COST_BPS,
                     'total_turnover': wf_tvr.sum(), 'total_costs': total_costs,
                     **{f'gross_{k}': v for k, v in m_gross.items()},
                     **{f'net_{k}': v for k, v in m_net.items()}})
summary.to_csv('data/model_input/cost_summary.csv')
print('Saved wf_net_returns.csv and cost_summary.csv')
print(summary.round(4).to_string())

## Summary

We charged a two-part toll — 2 bp commission + 5 bp slippage per side, applied to every pound that
moved, including the quarter-end forced closes — against the Phase 6 walk-forward, with a half/double
stress test around it. The headline write-up numbers are the **net Sharpe**, the **net cumulative
return**, and **costs as a share of gross profit**; the printed verdict sentence states practical
viability directly.